# GARCH — companion notebook

Companion for [`garch.md`](./garch.md). Simulate a GARCH(1,1) series, fit by maximum likelihood, and plot the recovered conditional volatility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
rng = np.random.default_rng(0)

In [ ]:
# Simulate GARCH(1,1)
T = 2000
omega_t, alpha_t, beta_t = 0.00002, 0.06, 0.92
sigma2 = np.empty(T); r = np.empty(T)
sigma2[0] = omega_t / (1 - alpha_t - beta_t)
for t in range(T):
    if t > 0:
        sigma2[t] = omega_t + alpha_t*r[t-1]**2 + beta_t*sigma2[t-1]
    r[t] = np.sqrt(sigma2[t]) * rng.standard_normal()

# MLE: negative log-likelihood
def nll(params, r):
    omega, alpha, beta = params
    if min(omega, alpha, beta) < 1e-10 or alpha + beta >= 0.999: return 1e10
    T = len(r); s2 = np.empty(T)
    s2[0] = np.var(r)
    for t in range(1, T):
        s2[t] = omega + alpha*r[t-1]**2 + beta*s2[t-1]
    return 0.5*np.sum(np.log(s2) + r**2 / s2)

res = minimize(nll, x0=[1e-5, 0.05, 0.9], args=(r,), method='Nelder-Mead', options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
omega_h, alpha_h, beta_h = res.x
print(f'True : omega={omega_t}, alpha={alpha_t}, beta={beta_t}')
print(f'MLE  : omega={omega_h:.2e}, alpha={alpha_h:.3f}, beta={beta_h:.3f}')
print(f'Persistence: true={alpha_t+beta_t:.3f}, fit={alpha_h+beta_h:.3f}')

In [ ]:
# Reconstruct fitted variance with MLE parameters
s2_fit = np.empty(T); s2_fit[0] = np.var(r)
for t in range(1, T):
    s2_fit[t] = omega_h + alpha_h*r[t-1]**2 + beta_h*s2_fit[t-1]

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(r, lw=0.7); axes[0].set_ylabel('returns'); axes[0].set_title('Simulated returns')
axes[1].plot(np.sqrt(sigma2), label='true volatility', lw=1.2)
axes[1].plot(np.sqrt(s2_fit), label='MLE-fit volatility', lw=1.2, alpha=0.7)
axes[1].set_ylabel('conditional sigma_t'); axes[1].set_xlabel('time'); axes[1].legend()
plt.tight_layout(); plt.show()